In [ ]:
# ── Config ──────────────────────────────────────────────────
import os; os.makedirs('output', exist_ok=True)

SWC_FILE  = 'output/FN1_01/neurons_auto.swc'
SOMA_NPZ  = 'output/FN1_01/soma.npz'
STACK_TIF = 'output/FN1_01/stack_preprocessed.tif'
PREP_NPZ  = 'output/FN1_01/prep_riem.npz'


In [ ]:
# ── Load ─────────────────────────────────────────────────────
import numpy as np
import plotly.graph_objects as go

swc_nodes = {}
with open(SWC_FILE) as f:
    for line in f:
        if line.startswith('#') or not line.strip(): continue
        p = line.split()
        nid = int(p[0])
        swc_nodes[nid] = dict(
            type=int(p[1]), x=float(p[2]), y=float(p[3]),
            z=float(p[4]), r=float(p[5]), parent=int(p[6])
        )

soma_data   = np.load(SOMA_NPZ)
sv_local    = soma_data['mesh_verts'].astype(np.float64)
sf          = soma_data['mesh_faces']
soma_cv     = soma_data['soma_centroid_vox']
voxel_iso_s = float(soma_data['voxel_iso'])

soma_global_um = soma_cv * voxel_iso_s
mesh_local_ctr = sv_local.mean(axis=0)
offset         = soma_global_um - mesh_local_ctr
sv             = sv_local + offset

print(f'SWC nodes  : {len(swc_nodes):,}')
print(f'Soma verts : {len(sv):,}  faces: {len(sf):,}')

In [ ]:
# ── Path length + branch order ─────────────────────────────
from collections import defaultdict

def dist_um(a, b):
    return np.sqrt((a['x']-b['x'])**2+(a['y']-b['y'])**2+(a['z']-b['z'])**2)

children_map = defaultdict(list)
for nid, n in swc_nodes.items():
    if n['parent'] != -1:
        children_map[n['parent']].append(nid)

path_len = {1: 0.0}
queue    = [1]
while queue:
    cur = queue.pop(0)
    for ch in children_map.get(cur, []):
        path_len[ch] = path_len[cur] + dist_um(swc_nodes[cur], swc_nodes[ch])
        queue.append(ch)

branch_order = {1: 0}
queue = [1]
while queue:
    cur  = queue.pop(0)
    kids = children_map.get(cur, [])
    for ch in kids:
        branch_order[ch] = branch_order[cur] + (1 if len(kids) >= 2 else 0)
        queue.append(ch)

print(f'Max path length : {max(path_len.values()):.1f} µm')
print(f'Max branch order: {max(branch_order.values())}')

In [ ]:
# ── 3D visualization: branch order color ────────────────────
from scipy.spatial import cKDTree

ORDER_COLORS = {
    0:'#888888', 1:'#FF4444', 2:'#FF8C00',
    3:'#FFD700', 4:'#44BB44', 5:'#4488FF',
}
def order_color(o): return ORDER_COLORS.get(min(o, 5), '#88AAFF')

soma_pts_xyz = np.column_stack([sv[:,2], sv[:,1], sv[:,0]])
soma_tree    = cKDTree(soma_pts_xyz)

def soma_surface_pt(node):
    _, idx = soma_tree.query([node['x'], node['y'], node['z']])
    return soma_pts_xyz[idx]

fig = go.Figure()

fig.add_trace(go.Mesh3d(
    x=sv[:,2], y=sv[:,1], z=sv[:,0],
    i=sf[:,0], j=sf[:,1], k=sf[:,2],
    color='#FF6B6B', opacity=0.9, flatshading=False,
    lighting=dict(ambient=0.5, diffuse=0.6, specular=0.3),
    lightposition=dict(x=2, y=2, z=5),
    name='Soma',
))

segs = defaultdict(lambda: {'x':[], 'y':[], 'z':[]})
for nid, n in swc_nodes.items():
    par_id = n['parent']
    if par_id == -1 or par_id not in swc_nodes: continue
    o = branch_order.get(nid, 0)
    if par_id == 1:
        sx, sy, sz = soma_surface_pt(n)
        segs[o]['x'] += [n['x'], sx, None]
        segs[o]['y'] += [n['y'], sy, None]
        segs[o]['z'] += [n['z'], sz, None]
    else:
        p = swc_nodes[par_id]
        segs[o]['x'] += [n['x'], p['x'], None]
        segs[o]['y'] += [n['y'], p['y'], None]
        segs[o]['z'] += [n['z'], p['z'], None]

ORDER_LABEL = {1:'1st (primary)',2:'2nd',3:'3rd',4:'4th'}
for o in sorted(segs.keys()):
    if o == 0: continue
    fig.add_trace(go.Scatter3d(
        x=segs[o]['x'], y=segs[o]['y'], z=segs[o]['z'],
        mode='lines',
        line=dict(color=order_color(o), width=2 if o<=2 else 1),
        name=ORDER_LABEL.get(o, f'{o}th'),
        opacity=0.9,
    ))

fig.update_layout(
    title=dict(text='Neuron Riemannian — branch order color', font=dict(color='white')),
    scene=dict(
        xaxis_title='X (µm)', yaxis_title='Y (µm)', zaxis_title='Z (µm)',
        bgcolor='#0a0a0a',
        xaxis=dict(backgroundcolor='#0a0a0a', gridcolor='#222', color='white'),
        yaxis=dict(backgroundcolor='#0a0a0a', gridcolor='#222', color='white'),
        zaxis=dict(backgroundcolor='#0a0a0a', gridcolor='#222', color='white'),
        aspectmode='data',
    ),
    paper_bgcolor='#111111', font_color='white',
    margin=dict(l=0, r=0, t=50, b=0),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(30,30,30,0.8)'),
)
fig.show()

In [ ]:
# ── MIP overlay: XY / XZ / YZ ──────────────────────────────
import matplotlib.pyplot as plt
import tifffile
from matplotlib.lines import Line2D

stack      = tifffile.imread(STACK_TIF).astype(np.float32)
stack_norm = stack / stack.max()
d_prep     = np.load(PREP_NPZ)
voxel_iso  = float(d_prep['voxel_iso'])

mip_xy = stack_norm.max(axis=0)
mip_xz = stack_norm.max(axis=1)
mip_yz = stack_norm.max(axis=2)
del stack_norm

soma_n = swc_nodes[1]

fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
fig.patch.set_facecolor('#111111')

views = [
    (axes[0,0], axes[1,0], mip_xy, 'x', 'y', 'X (px)', 'Y (px)', 'XY'),
    (axes[0,1], axes[1,1], mip_xz, 'x', 'z', 'X (px)', 'Z (px)', 'XZ'),
    (axes[0,2], axes[1,2], mip_yz, 'y', 'z', 'Y (px)', 'Z (px)', 'YZ'),
]

for ax_raw, ax_swc, mip, hk, vk, hl, vl, title in views:
    for ax in (ax_raw, ax_swc):
        ax.imshow(mip, cmap='gray', origin='upper', vmin=0, vmax=0.8, aspect='equal')
        ax.axis('off')
    ax_raw.set_title(f'Raw MIP — {title}', color='white', fontsize=10)
    ax_swc.set_title(f'SWC overlay — {title}', color='white', fontsize=10)
    for nid, n in swc_nodes.items():
        if n['parent'] == -1 or n['parent'] not in swc_nodes: continue
        p = swc_nodes[n['parent']]
        o = branch_order.get(nid, 0)
        ax_swc.plot([n[hk]/voxel_iso, p[hk]/voxel_iso],
                    [n[vk]/voxel_iso, p[vk]/voxel_iso],
                    '-', color=order_color(o), lw=0.6, alpha=0.75)
    ax_swc.scatter([soma_n[hk]/voxel_iso], [soma_n[vk]/voxel_iso],
                   c='white', s=60, marker='*', zorder=5)

legend_items = [
    Line2D([0],[0], color='#FF4444', lw=2, label='1st'),
    Line2D([0],[0], color='#FF8C00', lw=2, label='2nd'),
    Line2D([0],[0], color='#FFD700', lw=2, label='3rd'),
    Line2D([0],[0], color='#44BB44', lw=2, label='4th'),
    Line2D([0],[0], color='#4488FF', lw=2, label='5th+'),
]
axes[1,2].legend(handles=legend_items, facecolor='#222',
                 labelcolor='white', fontsize=8, loc='upper right')
fig.suptitle('MIP overlay (Riemannian) — XY / XZ / YZ', color='white', fontsize=12)
plt.show()

In [ ]:
# ── Loop detection ───────────────────────────────────────────
import sys; sys.setrecursionlimit(100000)

visited, in_stack, loop_nodes = set(), set(), []

def dfs(node):
    visited.add(node); in_stack.add(node)
    for ch in children_map.get(node, []):
        if ch not in visited: dfs(ch)
        elif ch in in_stack: loop_nodes.append((node, ch))
    in_stack.discard(node)

dfs(1)
isolated = set(swc_nodes.keys()) - visited
print(f'Nodes total    : {len(swc_nodes):,}')
print(f'Nodes connected: {len(visited):,}')
print(f'Nodes isolated : {len(isolated):,}')
print(f'Loops detected : {len(loop_nodes)}')
if not loop_nodes: print('  → tree structure OK')

In [ ]:
# ── Morphology ───────────────────────────────────────────────
from scipy.spatial import cKDTree

soma_pts_xyz = np.column_stack([sv[:,2], sv[:,1], sv[:,0]])
soma_tree    = cKDTree(soma_pts_xyz)
soma_c       = swc_nodes[1]

primary_correction = {}
for nid in children_map.get(1, []):
    n = swc_nodes[nid]
    _, idx = soma_tree.query([n['x'], n['y'], n['z']])
    sx, sy, sz = soma_pts_xyz[idx]
    primary_correction[nid] = np.sqrt(
        (sx-soma_c['x'])**2 + (sy-soma_c['y'])**2 + (sz-soma_c['z'])**2)

def get_primary_ancestor(nid):
    cur = nid
    while swc_nodes[cur]['parent'] not in (-1, 1):
        cur = swc_nodes[cur]['parent']
    return cur

def corrected_path_to_soma(tip_id):
    length, cur = 0.0, tip_id
    while swc_nodes[cur]['parent'] != -1:
        par = swc_nodes[cur]['parent']
        if par not in swc_nodes: break
        length += dist_um(swc_nodes[cur], swc_nodes[par])
        cur = par
    return max(length - primary_correction.get(get_primary_ancestor(tip_id), 0.0), 0.0)

connected   = visited
tips_ids    = [nid for nid in connected if not children_map.get(nid)]
branch_pts  = [nid for nid in connected if len(children_map.get(nid,[])) >= 2]
primary_ids = children_map.get(1, [])
tip_lengths = [corrected_path_to_soma(t) for t in tips_ids]

seg_raw          = sum(dist_um(swc_nodes[nid], swc_nodes[n['parent']])
                       for nid, n in swc_nodes.items()
                       if n['parent'] != -1 and n['parent'] in swc_nodes)
total_correction = sum(primary_correction.values())

d_prep    = np.load(PREP_NPZ)
soma_r_um = float(d_prep['soma_r_um'])

morph = dict(
    soma_radius_um   = soma_r_um,
    n_primary        = len(primary_ids),
    n_tips           = len(tips_ids),
    n_branch_pts     = len(branch_pts),
    max_branch_order = max(branch_order.values(), default=0),
    total_length_um  = seg_raw - total_correction,
    tip_dist_mean_um = float(np.mean(tip_lengths)) if tip_lengths else 0,
    tip_dist_max_um  = float(np.max(tip_lengths))  if tip_lengths else 0,
)

LABELS = {
    'soma_radius_um':'Soma radius','n_primary':'Primary branches',
    'n_tips':'Tips','n_branch_pts':'Branch points',
    'max_branch_order':'Max branch order','total_length_um':'Total length',
    'tip_dist_mean_um':'Tip dist mean','tip_dist_max_um':'Tip dist max',
}
print('='*48)
for k, v in morph.items():
    print(f'  {LABELS[k]:22s}: {v:.1f}')
print('='*48)

In [ ]:
# ── Morphology Summary — all parameters at a glance ─────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
import numpy as np

# ── Base computation (reusing morphology cell results) ──────
radii_all   = np.array([swc_nodes[n]['r'] for n in swc_nodes])
bos_all     = np.array([branch_order.get(n, 0) for n in swc_nodes])
bos_tips    = np.array([branch_order.get(t, 0) for t in tips_ids])
seg_lens    = [dist_um(swc_nodes[n], swc_nodes[swc_nodes[n]['parent']])
               for n in swc_nodes
               if swc_nodes[n]['parent'] != -1 and swc_nodes[n]['parent'] in swc_nodes]

# tip count per primary
primary_of_tip = {}
for t in tips_ids:
    cur = t
    while cur in swc_nodes and swc_nodes[cur]['parent'] not in (-1, 1):
        cur = swc_nodes[cur]['parent']
    primary_of_tip[t] = cur
tips_per_primary = Counter(primary_of_tip.values())

# ── Text summary ────────────────────────────────────────────
W = 52
print('╔' + '═'*W + '╗')
print(f'║{"  MORPHOLOGY SUMMARY":^{W}}║')
print('╠' + '═'*W + '╣')

def row(label, val, unit=''):
    s = f'  {label:<26}{val:<18}{unit}'
    print(f'║{s:<{W}}║')

def divider(title=''):
    if title:
        print(f'╠{"─"*W}╣')
        print(f'║  {title:<{W-2}}║')
        print(f'╠{"─"*W}╣')
    else:
        print(f'╠{"─"*W}╣')

divider('Soma')
row('Radius',         f'{soma_r_um:.2f}',      'µm')
row('Volume (sphere)',f'{4/3*np.pi*soma_r_um**3:.0f}', 'µm³')

divider('Structure')
row('Primary branches',  f'{len(primary_ids)}')
row('Branch points',     f'{len(branch_pts)}')
row('Tips',              f'{len(tips_ids)}')
row('Total nodes',       f'{len(swc_nodes):,}')

divider('Branch Order')
row('Max',               f'{int(bos_all.max())}')
row('Mean (all nodes)',  f'{bos_all.mean():.2f}')
row('Median (tips)',     f'{int(np.median(bos_tips))}')
row('Tips p75 order',    f'{int(np.percentile(bos_tips, 75))}')

divider('Path Length (soma surface corrected)')
tl = np.array(tip_lengths)
row('Total dendritic',   f'{morph["total_length_um"]:.0f}',  'µm')
row('Tip dist mean',     f'{tl.mean():.1f}',                 'µm')
row('Tip dist std',      f'{tl.std():.1f}',                  'µm')
row('Tip dist median',   f'{np.median(tl):.1f}',             'µm')
row('Tip dist max',      f'{tl.max():.1f}',                  'µm')
row('Tip dist min',      f'{tl.min():.1f}',                  'µm')

divider('Radius (EDT)')
row('Min',               f'{radii_all.min():.3f}',           'µm')
row('Median',            f'{np.median(radii_all):.3f}',      'µm')
row('Mean',              f'{radii_all.mean():.3f}',          'µm')
row('p99',               f'{np.percentile(radii_all,99):.3f}','µm')
row('Max',               f'{radii_all.max():.3f}',           'µm')

divider('Segment')
row('Mean seg length',   f'{np.mean(seg_lens):.3f}',         'µm')
row('Total segments',    f'{len(seg_lens):,}')

divider('Per-Primary (tips)')
pvals = sorted(tips_per_primary.values(), reverse=True)
row('Max tips/primary',  f'{pvals[0] if pvals else 0}')
row('Min tips/primary',  f'{pvals[-1] if pvals else 0}')
row('Mean tips/primary', f'{np.mean(pvals):.1f}' if pvals else '0')

print('╚' + '═'*W + '╝')

# ── Dashboard visualization ─────────────────────────────────
fig = plt.figure(figsize=(16, 10), facecolor='#111111')
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.4)

axes_list = [fig.add_subplot(gs[r,c]) for r in range(2) for c in range(4)]
for ax in axes_list:
    ax.set_facecolor('#1a1a1a'); ax.tick_params(colors='white')
    ax.spines[:].set_color('#444')

def ax_style(ax, title, xlabel='', ylabel=''):
    ax.set_title(title, color='white', fontsize=9)
    if xlabel: ax.set_xlabel(xlabel, color='white', fontsize=8)
    if ylabel: ax.set_ylabel(ylabel, color='white', fontsize=8)

# 0: tip distance distribution
axes_list[0].hist(tl, bins=25, color='steelblue')
axes_list[0].axvline(tl.mean(), color='orange', ls='--', lw=1.5,
                     label=f'mean {tl.mean():.0f}')
axes_list[0].legend(facecolor='#222', labelcolor='white', fontsize=7)
ax_style(axes_list[0], 'Tip distance dist.', 'µm')

# 1: branch order at tip
axes_list[1].hist(bos_tips, bins=range(int(bos_tips.max())+2),
                  color='coral', align='left')
ax_style(axes_list[1], 'Branch order @ tip', 'order')

# 2: tips per primary
axes_list[2].bar(range(len(pvals)), pvals, color='mediumpurple')
ax_style(axes_list[2], 'Tips per primary', 'primary (sorted)', '# tips')

# 3: radius distribution
axes_list[3].hist(radii_all[radii_all > 0.05], bins=40, color='#44BB44')
axes_list[3].axvline(np.median(radii_all), color='orange', ls='--', lw=1.5,
                     label=f'median {np.median(radii_all):.2f}')
axes_list[3].legend(facecolor='#222', labelcolor='white', fontsize=7)
ax_style(axes_list[3], 'Radius dist.', 'µm')

# 4: segment length
axes_list[4].hist(seg_lens, bins=40, color='#4488FF')
ax_style(axes_list[4], 'Segment length dist.', 'µm')

# 5: radius by branch order (boxplot)
max_o = min(int(bos_all.max()), 8)
bdata = [radii_all[bos_all == o] for o in range(max_o+1)]
bdata = [d for d in bdata if len(d) > 0]
if bdata:
    bp = axes_list[5].boxplot(bdata, patch_artist=True,
                              medianprops=dict(color='white', lw=1.5))
    for patch in bp['boxes']:
        patch.set_facecolor('#4488FF'); patch.set_alpha(0.6)
ax_style(axes_list[5], 'Radius by branch order', 'order', 'µm')

# 6: cumulative tip distance
axes_list[6].plot(np.sort(tl), np.linspace(0,1,len(tl)),
                  color='cyan', lw=2)
axes_list[6].axhline(0.5, color='#666', ls=':', lw=1)
ax_style(axes_list[6], 'CDF — tip distance', 'µm', 'cumulative')

# 7: branch order CDF
bo_vals = np.sort(bos_tips)
axes_list[7].plot(bo_vals, np.linspace(0,1,len(bo_vals)),
                  color='tomato', lw=2)
axes_list[7].axhline(0.5, color='#666', ls=':', lw=1)
ax_style(axes_list[7], 'CDF — branch order', 'order', 'cumulative')

fig.suptitle('Morphology Dashboard', color='white', fontsize=13, y=1.01)
plt.show()

In [ ]:
# ── 3D Tube Mesh: same primary → same color ─────────────────
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from collections import defaultdict

# ── Per-primary color palette ───────────────────────────────
primary_ids_list = sorted(children_map.get(1, []))
cmap   = plt.cm.tab20
PRIMARY_COLORS = {
    pid: f'rgb{tuple(int(c*255) for c in cmap(i / max(len(primary_ids_list), 1))[:3])}'
    for i, pid in enumerate(primary_ids_list)
}

def make_tube(node_ids, n_sides=6, sample=1):
    idx = list(range(0, len(node_ids), max(1, sample)))
    if idx[-1] != len(node_ids)-1: idx.append(len(node_ids)-1)
    nodes = [node_ids[i] for i in idx]
    if len(nodes) < 2: return None, None
    pts = np.array([[swc_nodes[n]['x'],swc_nodes[n]['y'],swc_nodes[n]['z']]
                    for n in nodes], dtype=np.float64)
    rs  = np.array([max(swc_nodes[n]['r'], 0.2) for n in nodes])
    rings = []; right = None
    for i,(p,r) in enumerate(zip(pts,rs)):
        t = pts[i+1]-pts[i] if i<len(pts)-1 else pts[i]-pts[i-1]
        tl = np.linalg.norm(t)
        t  = t/tl if tl>1e-8 else np.array([0.,0.,1.])
        if right is None:
            ref = np.array([0.,0.,1.]) if abs(t[2])<0.9 else np.array([1.,0.,0.])
            right = np.cross(t, ref)
        else:
            right = right - np.dot(right,t)*t
        rl = np.linalg.norm(right)
        right = np.cross(t, np.array([0.,0.,1.]) if abs(t[2])<0.9 else np.array([1.,0.,0.])) \
                if rl<1e-8 else right/rl
        up  = np.cross(t, right)
        ang = np.linspace(0, 2*np.pi, n_sides, endpoint=False)
        rings.append(p + r*(np.outer(np.cos(ang),right)+np.outer(np.sin(ang),up)))
    V = np.vstack(rings).astype(np.float32)
    F = []
    for i in range(len(rings)-1):
        for j in range(n_sides):
            k = (j+1)%n_sides
            v00,v01 = i*n_sides+j, i*n_sides+k
            v10,v11 = (i+1)*n_sides+j, (i+1)*n_sides+k
            F+=[[v00,v10,v01],[v01,v10,v11]]
    return V, np.array(F, dtype=np.int32)

def collect_segments():
    segs  = []
    stack = [(pid, [], pid) for pid in children_map.get(1,[])]
    while stack:
        node, path, panc = stack.pop()
        path = path+[node]
        kids = children_map.get(node,[])
        if not kids: segs.append((path, panc))
        elif len(kids)==1: stack.append((kids[0], path, panc))
        else:
            segs.append((path, panc))
            for kid in kids: stack.append((kid, [node], panc))
    return segs

segments = collect_segments()

# accumulate mesh per primary
mesh_verts  = defaultdict(list)
mesh_faces  = defaultdict(list)
mesh_offset = defaultdict(int)

for path_ids, panc in segments:
    order  = branch_order.get(path_ids[0], 1)
    color  = PRIMARY_COLORS.get(panc, 'rgb(150,150,150)')
    sides  = 8 if order <= 1 else 6 if order <= 3 else 4
    sample = 1 if order <= 1 else 2
    V, F   = make_tube(path_ids, n_sides=sides, sample=sample)
    if V is None: continue
    mesh_faces[color].append(F + mesh_offset[color])
    mesh_verts[color].append(V)
    mesh_offset[color] += len(V)

# ── Figure ────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Mesh3d(
    x=sv[:,2], y=sv[:,1], z=sv[:,0],
    i=sf[:,0], j=sf[:,1], k=sf[:,2],
    color='#FF6B6B', opacity=0.85, flatshading=False,
    lighting=dict(ambient=0.5,diffuse=0.7,specular=0.3),
    lightposition=dict(x=3,y=3,z=5), name='Soma',
))

for i, (color, vlist) in enumerate(mesh_verts.items()):
    if not vlist: continue
    V_all = np.vstack(vlist)
    F_all = np.vstack(mesh_faces[color])
    pid   = primary_ids_list[i] if i < len(primary_ids_list) else '?'
    fig.add_trace(go.Mesh3d(
        x=V_all[:,0], y=V_all[:,1], z=V_all[:,2],
        i=F_all[:,0], j=F_all[:,1], k=F_all[:,2],
        color=color, opacity=1.0, flatshading=False,
        lighting=dict(ambient=0.4,diffuse=0.7,specular=0.4),
        lightposition=dict(x=3,y=3,z=5),
        name=f'Primary {pid}',
    ))

fig.update_layout(
    title=dict(text='Neuron Tube Mesh — color: primary branch lineage', font=dict(color='white')),
    scene=dict(
        xaxis_title='X (µm)', yaxis_title='Y (µm)', zaxis_title='Z (µm)',
        bgcolor='#080808',
        xaxis=dict(backgroundcolor='#080808',gridcolor='#1a1a1a',color='white'),
        yaxis=dict(backgroundcolor='#080808',gridcolor='#1a1a1a',color='white'),
        zaxis=dict(backgroundcolor='#080808',gridcolor='#1a1a1a',color='white'),
        aspectmode='data',
    ),
    paper_bgcolor='#111111', font_color='white',
    margin=dict(l=0,r=0,t=50,b=0),
    legend=dict(x=0.01,y=0.99,bgcolor='rgba(20,20,20,0.85)'),
)
fig.show()
print(f'Primary branches: {len(primary_ids_list)}')

In [ ]:
# ── Dendrogram ───────────────────────────────────────────────
import matplotlib.pyplot as plt, sys
sys.setrecursionlimit(200000)

cmap_p  = plt.cm.tab20
p_color = {pid: cmap_p(i/max(len(primary_ids),1)) for i,pid in enumerate(primary_ids)}

fig, ax = plt.subplots(figsize=(14, max(4, len(primary_ids)*0.3+2)))
y_counter = [0]

def draw_branch(node_id, x_start, y_pos, color):
    cur, x = node_id, x_start
    while True:
        kids = children_map.get(cur, [])
        if not kids:
            ax.plot([x_start,x],[y_pos,y_pos],'-',color=color,lw=1.0,alpha=0.85)
            return
        if len(kids)>1:
            ax.plot([x_start,x],[y_pos,y_pos],'-',color=color,lw=1.0,alpha=0.85)
            ys = []
            for kid in sorted(kids):
                y_counter[0]+=1
                draw_branch(kid,x,y_counter[0],color)
                ys.append(y_counter[0])
            ax.plot([x,x],[min(ys),max(ys)],'-',color=color,lw=1.0,alpha=0.85)
            return
        nxt = kids[0]
        x  += dist_um(swc_nodes[cur],swc_nodes[nxt])
        cur = nxt

for pid in sorted(primary_ids):
    y_counter[0]+=1
    draw_branch(pid, 0, y_counter[0], p_color.get(pid,'white'))

ax.axvline(0, color='tomato', lw=2, label='Soma')
ax.set_xlabel('Distance from soma (µm)', color='white')
ax.set_ylabel('Branch index', color='white')
ax.set_title(f'Dendrogram — Riemannian ({len(primary_ids)} primaries)', color='white')
ax.set_facecolor('#111111'); fig.patch.set_facecolor('#111111')
ax.tick_params(colors='white'); ax.spines[:].set_color('#444')
ax.legend(facecolor='#222', labelcolor='white')
plt.tight_layout(); plt.show()

In [ ]:
# ── Branch distribution ─────────────────────────────────────
import matplotlib.pyplot as plt
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
fig.patch.set_facecolor('#111111')
for ax in axes:
    ax.set_facecolor('#111111'); ax.tick_params(colors='white'); ax.spines[:].set_color('#444')

axes[0].hist(tip_lengths, bins=30, color='steelblue', edgecolor='none')
axes[0].set_xlabel('Path length to soma (µm)', color='white')
axes[0].set_title(f'Tip distances (n={len(tips_ids)})', color='white')

bos = [branch_order.get(t, 0) for t in tips_ids]
axes[1].hist(bos, bins=range(max(bos)+2) if bos else 1,
             color='coral', edgecolor='none', align='left')
axes[1].set_xlabel('Branch order at tip', color='white')
axes[1].set_title('Branch order distribution', color='white')

primary_of_tip = {}
for t in tips_ids:
    cur = t
    while cur in swc_nodes and swc_nodes[cur]['parent'] not in (-1, 1):
        cur = swc_nodes[cur]['parent']
    primary_of_tip[t] = cur
tips_per_p = Counter(primary_of_tip.values())
vals = sorted(tips_per_p.values(), reverse=True)
axes[2].bar(range(len(vals)), vals, color='mediumpurple', edgecolor='none')
axes[2].set_xlabel('Primary branch (sorted)', color='white')
axes[2].set_ylabel('# Tips', color='white')
axes[2].set_title(f'Tips per primary ({len(primary_ids)} primaries)', color='white')

plt.show()

In [ ]:
# ── Radius colormap visualization ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import plotly.graph_objects as go

# ── Collect per-node radius ─────────────────────────────────
nids   = list(swc_nodes.keys())
radii  = np.array([swc_nodes[n]['r'] for n in nids])

r_min, r_max = radii.min(), np.percentile(radii, 99)
print(f'Radius  min={radii.min():.3f}  median={np.median(radii):.3f}  '
      f'p99={np.percentile(radii,99):.3f}  max={radii.max():.3f} µm')

# print primary node radii separately
primary_ids = children_map.get(1, [])
print(f'\nPrimary branch first-node radius:')
for pid in sorted(primary_ids):
    n = swc_nodes[pid]
    print(f'  pid {pid}: r={n["r"]:.3f} µm')

# ── Histogram ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
fig.patch.set_facecolor('#111111')
for ax in axes:
    ax.set_facecolor('#111111'); ax.tick_params(colors='white'); ax.spines[:].set_color('#444')

axes[0].hist(radii, bins=60, color='steelblue', edgecolor='none')
axes[0].axvline(np.median(radii), color='orange', ls='--', label=f'median {np.median(radii):.2f} µm')
axes[0].axvline(0.5, color='red', ls=':', label='0.5 µm')
axes[0].set_xlabel('Radius (µm)', color='white')
axes[0].set_title('All-node radius distribution', color='white')
axes[0].legend(facecolor='#222', labelcolor='white', fontsize=8)

# radius boxplot by branch order
orders = [branch_order.get(n, 0) for n in nids]
max_o  = min(max(orders), 8)
data   = [radii[np.array(orders) == o] for o in range(max_o+1)]
bp = axes[1].boxplot(data, patch_artist=True,
                     medianprops=dict(color='white', lw=2))
for patch, o in zip(bp['boxes'], range(max_o+1)):
    patch.set_facecolor(order_color(o))
    patch.set_alpha(0.7)
axes[1].set_xlabel('Branch order', color='white')
axes[1].set_ylabel('Radius (µm)', color='white')
axes[1].set_title('Radius by branch order', color='white')
axes[1].set_xticklabels(range(max_o+1))
plt.show()

# ── MIP overlay: radius colormap ────────────────────────────
import tifffile
stack      = tifffile.imread(STACK_TIF).astype(np.float32)
voxel_iso  = float(np.load(PREP_NPZ)['voxel_iso'])
mip_xy     = stack.max(axis=0) / stack.max()
del stack

cmap_r = cm.get_cmap('RdYlGn')   # thin=red, thick=green

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
fig.patch.set_facecolor('#111111')
ax.imshow(mip_xy, cmap='gray', origin='upper', vmin=0, vmax=0.8, aspect='equal')
ax.axis('off')

for nid, n in swc_nodes.items():
    par_id = n['parent']
    if par_id == -1 or par_id not in swc_nodes: continue
    p = swc_nodes[par_id]
    r_norm = np.clip((n['r'] - r_min) / (r_max - r_min + 1e-8), 0, 1)
    color  = cmap_r(r_norm)
    ax.plot([n['x']/voxel_iso, p['x']/voxel_iso],
            [n['y']/voxel_iso, p['y']/voxel_iso],
            '-', color=color, lw=0.8, alpha=0.8)

# colorbar
sm = plt.cm.ScalarMappable(cmap=cmap_r,
     norm=plt.Normalize(vmin=r_min, vmax=r_max))
cb = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cb.set_label('Radius (µm)', color='white')
cb.ax.tick_params(colors='white')
ax.set_title('Radius colormap — XY MIP  (red=thin, green=thick)', color='white')
fig.patch.set_facecolor('#111111')
plt.show()

In [ ]:
# ── Axon / Apical tuft estimation ──────────────────────────────
# Axon  : min(mean_r) — thinnest primary (most reliable single criterion)
# Apical: dir_z < -0.1 (pia direction), max_path largest — longest branch extending upward
# Basal : all others

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from collections import defaultdict
from matplotlib.patches import Patch

# ── Per-primary statistics ───────────────────────────────────
d_prep = np.load(PREP_NPZ)

def get_subtree(root_id):
    result, stack = [], [root_id]
    while stack:
        cur = stack.pop(); result.append(cur)
        stack.extend(children_map.get(cur, []))
    return result

soma_pos = np.array([swc_nodes[1]['z'], swc_nodes[1]['y'], swc_nodes[1]['x']])

per_primary = {}
for pid in primary_ids:
    subtree      = get_subtree(pid)
    subtree_tips = [n for n in subtree if not children_map.get(n)]
    radii        = np.array([swc_nodes[n]['r'] for n in subtree])
    plens        = [corrected_path_to_soma(t) for t in subtree_tips]

    farthest = max(subtree_tips, key=lambda t: path_len.get(t, 0)) if subtree_tips else pid
    fn  = swc_nodes[farthest]
    vec = np.array([fn['z']-soma_pos[0], fn['y']-soma_pos[1], fn['x']-soma_pos[2]])
    vec_norm = vec / (np.linalg.norm(vec) + 1e-8)

    tip_dirs = []
    for t in subtree_tips:
        tn = swc_nodes[t]
        v  = np.array([tn['z']-soma_pos[0], tn['y']-soma_pos[1], tn['x']-soma_pos[2]])
        vl = np.linalg.norm(v)
        if vl > 1: tip_dirs.append(v / vl)
    dir_consistency = float(np.mean([abs(np.dot(d, vec_norm)) for d in tip_dirs])) \
                      if tip_dirs else 0.0

    per_primary[pid] = {
        'n_tips'      : len(subtree_tips),
        'mean_r'      : float(radii.mean()),
        'std_r'       : float(radii.std()),
        'max_path'    : max(plens) if plens else 0.0,
        'dir_vec'     : vec_norm,
        'dir_z'       : float(vec_norm[0]),   # negative=pia (up), positive=WM (down)
        'dir_consist' : dir_consistency,
    }

pids = list(per_primary.keys())

# ── classification ─────────────────────────────────────────────────────
# 1. Axon: thinnest primary
axon_pid = min(pids, key=lambda p: per_primary[p]['mean_r'])

# 2. Apical: excluding axon, largest max_path among dir_z < -0.1 (pia direction)
pia_pids = [p for p in pids if p != axon_pid and per_primary[p]['dir_z'] < -0.1]
apical_pid = max(pia_pids, key=lambda p: per_primary[p]['max_path']) if pia_pids else None

labels = {}
for p in pids:
    if p == axon_pid:     labels[p] = 'Axon'
    elif p == apical_pid: labels[p] = 'Apical'
    else:                 labels[p] = 'Basal'

CLASSIFY_COLORS = {'Axon':'#FFD700', 'Apical':'#00CFFF', 'Basal':'#FF6666'}

# ── Text output ─────────────────────────────────────────────────
print('=' * 72)
print(f'  {"PID":>5} {"Label":>7} {"mean_r":>7} {"std_r":>6} {"n_tips":>7} '
      f'{"max_path":>9} {"dir_z":>7} {"dir_z<-0.1":>10}')
print('-' * 72)
for p in sorted(pids):
    s   = per_primary[p]
    lbl = labels[p]
    mrk = ' ◀' if lbl != 'Basal' else ''
    pia = '  pia▲' if s['dir_z'] < -0.1 else ''
    print(f'  {p:>5} {lbl:>7} {s["mean_r"]:>7.3f} {s["std_r"]:>6.3f} '
          f'{s["n_tips"]:>7} {s["max_path"]:>9.1f} {s["dir_z"]:>7.3f}{pia}{mrk}')
print('=' * 72)

s_axon = per_primary[axon_pid]
print(f'\n[Axon]   pid={axon_pid}')
print(f'  mean_r   = {s_axon["mean_r"]:.3f} µm  ← minimum among all primaries')
print(f'  std_r    = {s_axon["std_r"]:.3f} µm')
print(f'  n_tips   = {s_axon["n_tips"]}  (may include collaterals)')
print(f'  max_path = {s_axon["max_path"]:.1f} µm')
print(f'  dir_z    = {s_axon["dir_z"]:.3f}  (positive=WM direction)')

if apical_pid:
    s_ap = per_primary[apical_pid]
    print(f'\n[Apical] pid={apical_pid}')
    print(f'  dir_z    = {s_ap["dir_z"]:.3f}  ← longest among pia-direction (negative) branches')
    print(f'  max_path = {s_ap["max_path"]:.1f} µm  ← maximum among pia-direction branches')
    print(f'  n_tips   = {s_ap["n_tips"]}  (oblique branches + tuft)')
    print(f'  mean_r   = {s_ap["mean_r"]:.3f} µm')
    print(f'\n  Pia-direction candidates (dir_z < -0.1):')
    for p in sorted(pia_pids, key=lambda p: -per_primary[p]['max_path']):
        s = per_primary[p]
        tag = ' ← selected' if p == apical_pid else ''
        print(f'    pid={p}  dir_z={s["dir_z"]:.3f}  max_path={s["max_path"]:.1f} µm'
              f'  n_tips={s["n_tips"]}{tag}')
else:
    print('\n[Apical] no pia-direction branch found (dir_z < -0.1)')

# ── Scatter plot ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
fig.patch.set_facecolor('#111111')
for ax in axes:
    ax.set_facecolor('#1a1a1a'); ax.tick_params(colors='white'); ax.spines[:].set_color('#444')

for pid in pids:
    s   = per_primary[pid]
    col = CLASSIFY_COLORS[labels[pid]]
    axes[0].scatter(s['mean_r'], s['std_r'], color=col, s=90, zorder=3)
    axes[0].annotate(str(pid), (s['mean_r'], s['std_r']),
                     fontsize=7, color='white', xytext=(3,3), textcoords='offset points')
    axes[1].scatter(s['n_tips'], s['max_path'], color=col, s=90, zorder=3)
    axes[1].annotate(str(pid), (s['n_tips'], s['max_path']),
                     fontsize=7, color='white', xytext=(3,3), textcoords='offset points')
    axes[2].scatter(s['dir_z'], s['max_path'], color=col, s=90, zorder=3)
    axes[2].annotate(str(pid), (s['dir_z'], s['max_path']),
                     fontsize=7, color='white', xytext=(3,3), textcoords='offset points')

axes[0].set_xlabel('Mean radius (µm)', color='white')
axes[0].set_ylabel('Std radius (µm)', color='white')
axes[0].set_title('Radius\n(thinnest → Axon)', color='white')

axes[1].set_xlabel('Tip count', color='white')
axes[1].set_ylabel('Max path (µm)', color='white')
axes[1].set_title('Complexity vs Length', color='white')

axes[2].axvline(-0.1, color='#00CFFF', ls='--', lw=1.5, alpha=0.7, label='pia threshold')
axes[2].set_xlabel('dir_z  (negative=pia▲, positive=WM▼)', color='white')
axes[2].set_ylabel('Max path (µm)', color='white')
axes[2].set_title('Direction vs Length\n(pia direction + longest → Apical)', color='white')
axes[2].legend(facecolor='#222', labelcolor='white', fontsize=8)

legend_el = [Patch(color=v, label=k) for k, v in CLASSIFY_COLORS.items()]
axes[0].legend(handles=legend_el, facecolor='#222', labelcolor='white', fontsize=8)
plt.show()

# ── 3D Plotly ─────────────────────────────────────────────────
fig3d = go.Figure()
fig3d.add_trace(go.Mesh3d(
    x=sv[:,2], y=sv[:,1], z=sv[:,0],
    i=sf[:,0], j=sf[:,1], k=sf[:,2],
    color='#FF6B6B', opacity=0.7, name='Soma',
    lighting=dict(ambient=0.5, diffuse=0.6),
))

segs_cls = defaultdict(lambda: dict(x=[], y=[], z=[]))
for nid, n in swc_nodes.items():
    par = n['parent']
    if par == -1 or par not in swc_nodes: continue
    p   = swc_nodes[par]
    cur = nid
    while cur in swc_nodes and swc_nodes[cur]['parent'] not in (-1, 1):
        cur = swc_nodes[cur]['parent']
    lbl = labels.get(cur, 'Basal')
    segs_cls[lbl]['x'] += [n['x'], p['x'], None]
    segs_cls[lbl]['y'] += [n['y'], p['y'], None]
    segs_cls[lbl]['z'] += [n['z'], p['z'], None]

for lbl, segs in segs_cls.items():
    fig3d.add_trace(go.Scatter3d(
        **segs, mode='lines',
        line=dict(color=CLASSIFY_COLORS[lbl], width=3 if lbl != 'Basal' else 1),
        name=lbl, opacity=0.95,
    ))

fig3d.update_layout(
    title=dict(text='Axon (gold) / Apical (sky blue) / Basal (red)  —  Z↑=pia, Z↓=WM',
               font=dict(color='white')),
    scene=dict(
        bgcolor='#080808',
        xaxis=dict(backgroundcolor='#080808', gridcolor='#1a1a1a', color='white', title='X µm'),
        yaxis=dict(backgroundcolor='#080808', gridcolor='#1a1a1a', color='white', title='Y µm'),
        zaxis=dict(backgroundcolor='#080808', gridcolor='#1a1a1a', color='white', title='Z µm (↓=WM)'),
        aspectmode='data',
    ),
    paper_bgcolor='#111111', font_color='white',
    margin=dict(l=0, r=0, t=50, b=0),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(20,20,20,0.85)'),
)
fig3d.show()

In [ ]:
# ── Axon / Apical detailed parameters ────────────────────────
import numpy as np
import matplotlib.pyplot as plt

def subtree_node_radii(root_id):
    return np.array([swc_nodes[n]['r'] for n in get_subtree(root_id)])

def subtree_tip_lengths(root_id):
    tips = [n for n in get_subtree(root_id) if not children_map.get(n)]
    return np.array([corrected_path_to_soma(t) for t in tips]), tips

def segment_lengths_subtree(root_id):
    result = []
    for nid in get_subtree(root_id):
        par = swc_nodes[nid]['parent']
        if par != -1 and par in swc_nodes:
            result.append(dist_um(swc_nodes[nid], swc_nodes[par]))
    return np.array(result)

W = 52
def section(title):
    print(f'\n╠{"─"*W}╣')
    print(f'║  {title:<{W-2}}║')
    print(f'╠{"─"*W}╣')

def row(label, val, unit=''):
    s = f'  {label:<28}{str(val):<16}{unit}'
    print(f'║{s:<{W}}║')

for target_pid, kind in [(axon_pid, 'AXON'), (apical_pid, 'APICAL DENDRITE')]:
    if target_pid is None:
        continue

    radii   = subtree_node_radii(target_pid)
    tlens, tips = subtree_tip_lengths(target_pid)
    seglens = segment_lengths_subtree(target_pid)
    s       = per_primary[target_pid]

    # Apical only: trunk length (longest straight segment before branching)
    trunk_len = 0.0
    if kind == 'APICAL DENDRITE':
        cur = target_pid
        while True:
            kids = children_map.get(cur, [])
            if len(kids) != 1: break
            trunk_len += dist_um(swc_nodes[cur], swc_nodes[kids[0]])
            cur = kids[0]

    subtree_all = get_subtree(target_pid)
    branch_pts  = [n for n in subtree_all if len(children_map.get(n,[])) >= 2]
    max_bo      = max((branch_order.get(n,0) for n in subtree_all), default=0)

    print('╔' + '═'*W + '╗')
    print(f'║  {kind:<{W-2}}║')
    print('╠' + '═'*W + '╣')

    row('Primary ID',        target_pid)
    section('Morphology')
    row('Total length',      f'{seglens.sum():.1f}',   'µm')
    row('Max path (soma surf)', f'{tlens.max():.1f}',  'µm')
    row('Mean path',         f'{tlens.mean():.1f}',    'µm')
    row('Tips',              len(tips))
    row('Branch points',     len(branch_pts))
    row('Max branch order',  max_bo)

    if kind == 'APICAL DENDRITE' and trunk_len > 0:
        section('Apical trunk')
        row('Trunk length',  f'{trunk_len:.1f}',       'µm')
        oblique = len([n for n in subtree_all
                       if len(children_map.get(swc_nodes[n]['parent'],[]))>=2
                       and branch_order.get(n,0)==1])
        row('Oblique branches', oblique)
        tuft_tips = [t for t in tips if path_len.get(t,0) > trunk_len * 0.7]
        row('Tuft tips (distal 30%)', len(tuft_tips))

    section('Radius (µm)')
    row('Min',               f'{radii.min():.3f}',     'µm')
    row('Median',            f'{np.median(radii):.3f}','µm')
    row('Mean',              f'{radii.mean():.3f}',    'µm')
    row('Max',               f'{radii.max():.3f}',     'µm')
    row('Std',               f'{radii.std():.3f}',     'µm')
    row('CV (std/mean)',     f'{radii.std()/radii.mean():.3f}', '(lower = more uniform)')

    section('Segment')
    row('Mean seg length',   f'{seglens.mean():.3f}',  'µm')
    row('Segments total',    len(seglens))

    section('Direction')
    row('dir_z',             f'{s["dir_z"]:.3f}',
        '  ← pia(negative) / WM(positive)')
    row('dir_consistency',   f'{s["dir_consist"]:.3f}',
        '  (1=perfectly straight)')

    print('╚' + '═'*W + '╝')

# ── Visualization: Axon vs Apical radius profile ────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
fig.patch.set_facecolor('#111111')

for ax, (target_pid, kind, col) in zip(axes, [
    (axon_pid,   'Axon',   '#FFD700'),
    (apical_pid, 'Apical', '#00CFFF'),
]):
    if target_pid is None:
        ax.set_visible(False); continue
    ax.set_facecolor('#1a1a1a'); ax.tick_params(colors='white'); ax.spines[:].set_color('#444')

    # sort nodes by distance from soma → observe radius tapering
    subtree = get_subtree(target_pid)
    dist_r  = [(path_len.get(n, 0), swc_nodes[n]['r']) for n in subtree if n != 1]
    dist_r.sort()
    if dist_r:
        ds, rs = zip(*dist_r)
        ax.scatter(ds, rs, s=2, alpha=0.5, color=col)
        # rolling mean
        from numpy.lib.stride_tricks import sliding_window_view
        if len(rs) > 20:
            w   = min(50, len(rs)//5)
            rm  = np.convolve(rs, np.ones(w)/w, mode='valid')
            rdm = np.array(ds[w//2: w//2+len(rm)])
            ax.plot(rdm, rm, color='white', lw=1.5, alpha=0.8, label='rolling mean')
            ax.legend(facecolor='#222', labelcolor='white', fontsize=7)
    ax.set_xlabel('Distance from soma (µm)', color='white')
    ax.set_ylabel('Radius (µm)', color='white')
    ax.set_title(f'{kind} (pid={target_pid}) — radius taper', color='white')

plt.show()